Tema 2 - Perceptron

In [60]:
import numpy as np
import math

train = np.load("./nn-2026-fii-homework-1/train.npz")
test = np.load("./nn-2026-fii-homework-1/test.npz")

X_train = train["data"]
y_train = train["labels"]
X_test = test["data"]

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)

print(y_train)

X_train = X_train / 255.0
X_test = X_test / 255.0


W = np.random.randn(784, 10) * 0.01
b = np.zeros(10)

(60000, 784)
(60000,)
(15000, 784)
[9 3 6 ... 9 5 7]


Calculam Z-ul (care va fi o matrice de (m,10), fiind scorul pentru fiecare 10 perceptroni la toate imaginile), si apoi il trecem prin softmax pentru a afla Y_prob (care va fi o matrice tot de (m,10), de data asta fiind probabilitatile pentru fiecare valoare 0-9, pentru fiecare imagine)

In [61]:
def softmax(Z:np.array) -> np.array:
    Y = np.zeros((Z.shape[0],10))
    # line = 0
    # column = 0
    # for row in Z:
    #     ez_list = []
    #     for z in row:
    #         ez_list.append(math.exp(z))
    #     sum_ez = sum(ez_list)
    #     for z in row:
    #         yi = ez_list[column]/sum_ez
    #         Y[line][column] = yi
    #         column += 1
    #     line += 1
    #     column = 0
    
    exp_Z = np.exp(Z)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)
    return Y

# Y_hat va fi matrice de (m,1) si va stabili rezultatatele prezise la fiecare imagine
def predicted_class_matrix(Y:np.array) -> np.array:
    Y_hat = np.zeros((Y.shape[0],1))
    Y_hat = np.argmax(Y,axis=1,keepdims=True)
    return Y_hat
# Z will be of size (m,10)
def forward_propagation(X:np.array, W:np.array, b:np.array) -> np.array:
    Z = np.dot(X,W) + b
    # print(Z.shape)
    # print(Z)
    Y = softmax(Z)
    return Y

Y_prob = forward_propagation(X_train, W, b)
print(Y_prob)

Y_predicted = predicted_class_matrix(Y_prob)
print()
print(Y_predicted)


[[0.07976057 0.09578243 0.11455387 ... 0.10576687 0.10514734 0.10322884]
 [0.08458764 0.09805787 0.10611896 ... 0.09908594 0.09978336 0.09960309]
 [0.09902768 0.10886076 0.10093967 ... 0.09521976 0.10024664 0.09107153]
 ...
 [0.08209254 0.09459532 0.11328321 ... 0.1104217  0.10155368 0.10077601]
 [0.09495282 0.09594689 0.09862652 ... 0.10248973 0.10054762 0.09259165]
 [0.09847366 0.086922   0.10088274 ... 0.10233216 0.09973004 0.09962575]]

[[2]
 [4]
 [3]
 ...
 [2]
 [4]
 [5]]


Calculam CrossEntropy pentru a vedea eroarea. cross_entropy va returna eroarea calculata pentru toti cei 10 perceptroni (care este suma din produs vector target (care ar fi [0,0,1,0,...,0]) cu log(probabilitatea prezicerii de pe acel perceptron))

CrossEntropy este, pentru o imagine, suma din elementul din target(adica daca e cel prezis sau nu)*log(probabilitatea prezisa la acel index).

Dar pentru toate elementele mai putin target-ul meu corect, elementele din suma sunt zero, deci cross entropy ar fi -log(probabilitate_prezicere_corecta)

iar crossentropy pe tot setul vine -log(probabilitate_prezicere_corecta)/m

In [62]:
#putem sa ne folosim direct de y_train care e de forma [9,3,2,...., etc.] pentru a extrage indexul probabilitatii prezicerii corecte rapid
def cross_entropy(Y_prob:np.array, y_train:np.array)->np.array:
    m = y_train.shape[0]
    ## extrag valorile din y_prob de la 0 la m-1 la indicii din y_train
    probabilitati_corecte = Y_prob[np.arange(m), y_train]
    loss = -np.sum(np.log(probabilitati_corecte)) / m
    return loss

loss = cross_entropy(Y_prob, y_train)
print(loss)
    

2.321593141535103


Trecem la gradient descend. Vom calcula target, care este o matrice, fiecare linie avand forma [0,1,0,0,...,0], daca predictia mea ar trebui sa dea 2

Target-y va fi aceasta matrice minus matricea probabilitatilor, si da tot o matrice (m,10)

In [63]:
def get_target(y_train:np.array) -> np.array:
    Target = np.zeros((y_train.shape[0],10))
    # line=0
    # column=0
    # for label in y_train:
    #     for column in range(0,10):
    #         if column == label:
    #             Target[line][column] = 1
    #         else:
    #             Target[line][column] = 0
    #     line += 1

    Target[np.arange(y_train.shape[0]), y_train] = 1
    return Target

def gradient_descend(Target:np.array, Y_prob:np.array, learning_rate:float, X_train:np.array, W:np.array, b:np.array):
    gradient = Target - Y_prob
    transpose = np.transpose(X_train)

    m = X_train.shape[0]
    change_w = (learning_rate * np.dot(transpose,gradient))/m
    W = W + change_w
    change_b = (learning_rate * np.sum(gradient, axis=0))/m
    b = b + change_b
    return (W,b)

print(W)
print()
print(b)

Target = get_target(y_train)
(W,b) = gradient_descend(Target,Y_prob,0.01,X_train,W,b)

print(W)
print()

print(b)


[[ 0.01230162  0.00053521  0.00817395 ... -0.00177705  0.01426551
  -0.00757795]
 [-0.00135766  0.00044119 -0.00301551 ... -0.00694071 -0.00231255
  -0.01092148]
 [-0.00726588  0.00575796  0.00541579 ... -0.00088791  0.0078012
  -0.01189436]
 ...
 [-0.01453957 -0.00791698  0.01023412 ... -0.00923215 -0.00426944
  -0.00195853]
 [-0.0066632   0.00771837 -0.00583692 ... -0.00573453  0.00696822
  -0.00128564]
 [ 0.00182418 -0.00432458  0.00850597 ... -0.00596858  0.00329218
   0.00306424]]

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[[ 0.01208351  0.00040896  0.00879458 ... -0.0016839   0.01472424
  -0.0075601 ]
 [-0.00135678  0.00044173 -0.00301569 ... -0.00694089 -0.00231272
  -0.01092165]
 [-0.00732884  0.00569703  0.00533849 ... -0.00092061  0.00782398
  -0.01189498]
 ...
 [-0.01446502 -0.00786976  0.00990351 ... -0.00929777 -0.00425951
  -0.00200809]
 [-0.00671629  0.00768363 -0.00590132 ... -0.00574228  0.00699771
  -0.00126982]
 [ 0.00182114 -0.00433056  0.00850108 ... -0.00597419  0.00332643


Acum trecem la loop de antrenare

In [64]:
W = np.random.randn(784, 10) * 0.01
b = np.zeros(10)

epoches = 100
learning_rate = 0.1
batch_size = 128
Target = get_target(y_train)

print("Starting training...")

for epoch in range(0,epoches+1):

    permutation = np.random.permutation(X_train.shape[0])

    X_shuffled = X_train[permutation]
    Y_shuffled = y_train[permutation]

    for i in range(0,X_train.shape[0],batch_size):
        X_batch = X_shuffled[i:i+batch_size]
        Y_batch = Y_shuffled[i:i+batch_size]

        Target_batch = get_target(Y_batch)

        Y_prob_batch = forward_propagation(X_batch,W,b)
        W, b = gradient_descend(Target_batch, Y_prob_batch, learning_rate, X_batch, W, b)
    #printuri generale

    if epoch % 10 == 0:
        Y_prob_full = forward_propagation(X_train, W, b)
        loss = cross_entropy(Y_prob_full, y_train)
        Y_hat = predicted_class_matrix(Y_prob_full)
        accuracy = np.mean(Y_hat.flatten() == y_train)
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Accuracy: {accuracy * 100:.2f}%")

print("Finished training")

Starting training...
Epoch 000 | Loss: 0.4004 | Accuracy: 89.30%
Epoch 010 | Loss: 0.2847 | Accuracy: 92.08%
Epoch 020 | Loss: 0.2688 | Accuracy: 92.52%
Epoch 030 | Loss: 0.2611 | Accuracy: 92.78%
Epoch 040 | Loss: 0.2567 | Accuracy: 92.89%
Epoch 050 | Loss: 0.2536 | Accuracy: 93.00%
Epoch 060 | Loss: 0.2512 | Accuracy: 93.13%
Epoch 070 | Loss: 0.2485 | Accuracy: 93.19%
Epoch 080 | Loss: 0.2466 | Accuracy: 93.23%
Epoch 090 | Loss: 0.2453 | Accuracy: 93.29%
Epoch 100 | Loss: 0.2431 | Accuracy: 93.29%
Finished training


Testam datele

In [65]:
import pandas as pd

Y_prob_test = forward_propagation(X_test, W, b)

Y_predicted_test = predicted_class_matrix(Y_prob_test)
submission = pd.DataFrame({
    "ID": range(len(Y_predicted_test)),
    "target": Y_predicted_test.flatten()
})

submission.to_csv("submission.csv", index=False)
print("Submission file created")


Submission file created
